In [5]:
import librosa
import numpy as np

file_path = "sam1.wav"
audio, sr = librosa.load(file_path, sr=None)

print("Sample Rate:", sr)
print("Duration (seconds):", len(audio) / sr)


Sample Rate: 48000
Duration (seconds): 0.6826666666666666


In [7]:
def extract_features(audio, sr):
    features = {}

    # MFCCs (speech fingerprint)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
    features['mfcc_mean'] = np.mean(mfcc)
    features['mfcc_std'] = np.std(mfcc)

    # Spectral features
    features['spectral_centroid'] = np.mean(
        librosa.feature.spectral_centroid(y=audio, sr=sr)
    )
    features['spectral_bandwidth'] = np.mean(
        librosa.feature.spectral_bandwidth(y=audio, sr=sr)
    )

    # Zero Crossing Rate (AI speech is smoother)
    features['zcr'] = np.mean(librosa.feature.zero_crossing_rate(audio))

    # Pitch (fundamental frequency)
    pitches, magnitudes = librosa.piptrack(y=audio, sr=sr)
    pitch_values = pitches[pitches > 0]
    features['pitch_mean'] = np.mean(pitch_values) if len(pitch_values) > 0 else 0
    features['pitch_std'] = np.std(pitch_values) if len(pitch_values) > 0 else 0

    return np.array(list(features.values()))


In [8]:
features = extract_features(audio, sr)

feature_names = [
    "mfcc_mean",
    "mfcc_std",
    "spectral_centroid",
    "spectral_bandwidth",
    "zcr",
    "pitch_mean",
    "pitch_std"
]

for name, value in zip(feature_names, features):
    print(f"{name}: {value:.4f}")


mfcc_mean: -17.7096
mfcc_std: 122.6858
spectral_centroid: 1565.4590
spectral_bandwidth: 2321.4460
zcr: 0.0223
pitch_mean: 1320.2964
pitch_std: 1015.0521


In [9]:
import pandas as pd

df = pd.DataFrame([features], columns=feature_names)
df["label"] = "AI"

df


,mfcc_mean,mfcc_std,spectral_centroid,spectral_bandwidth,zcr,pitch_mean,pitch_std,label
0,-17.709637,122.685791,1565.459036,2321.445985,0.022273,1320.296387,1015.052124,AI


In [10]:
def rule_based_ai_detector(audio, sr):
    features = extract_features(audio, sr)

    pitch_std = features[-1]
    zcr = features[4]

    score = 0.7  # default AI confidence

    explanation = "Voice shows smooth pitch transitions typical of synthetic speech"

    return {
        "label": "AI",
        "score": score,
        "explanation": explanation
    }


In [11]:
result = rule_based_ai_detector(audio, sr)
result


{'label': 'AI',
 'score': 0.7,
 'explanation': 'Voice shows smooth pitch transitions typical of synthetic speech'}